[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/spatialft/spatialft.github.io/blob/main/notebooks/03_finetune.ipynb)

# Notebook 3 — Fine-Tuning with Unsloth

LoRA fine-tune LFM2.5-1.2B-Thinking on StepGame spatial reasoning data.

In [ ]:
import os, sys
REPO = '/content/spatialft.github.io'
if not os.path.exists(REPO):
    !git clone https://github.com/spatialft/spatialft.github.io.git {REPO}
os.chdir(f'{REPO}/notebooks')
if REPO not in sys.path:
    sys.path.insert(0, REPO)


In [ ]:
# Colab: install dependencies
!pip install -q -r ../requirements.txt


In [ ]:
import json
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import get_peft_model, LoraConfig, TaskType
from trl import SFTTrainer, SFTConfig

from src.dataset import SYSTEM_PROMPT


In [ ]:
MODEL_ID       = 'LiquidAI/LFM2.5-1.2B-Thinking'
MAX_SEQ_LENGTH = 512
LORA_RANK      = 16
OUTPUT_DIR     = '../results/finetuned/checkpoint'

In [ ]:
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

peft_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_RANK * 2,
    target_modules="all-linear",  # auto-detects LFM2.5 linear layers
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(base_model, peft_config)
model.print_trainable_parameters()


In [ ]:
with open('../data/processed/train_formatted.json') as f:
    train_data = json.load(f)

dataset = Dataset.from_list(train_data)
print(f'Training on {len(dataset)} examples')
print(dataset[0]['full_text'][:300])

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    max_seq_length=MAX_SEQ_LENGTH,  # must set here — LFM2.5 context is 32k, default causes OOM
    args=SFTConfig(
        dataset_text_field="full_text",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        warmup_steps=50,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=20,
        output_dir=OUTPUT_DIR,
        save_strategy="epoch",
        optim="paged_adamw_8bit",
        seed=42,
    ),
)

trainer.train()


In [ ]:
# Save LoRA adapter
model.save_pretrained('../results/finetuned/lora_adapter')
tokenizer.save_pretrained('../results/finetuned/lora_adapter')
print('Adapter saved.')